In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd '/content/drive/MyDrive/Colab Notebooks/Rag project'

/content/drive/MyDrive/Colab Notebooks/Rag project


In [ ]:
import gc
import json
import os
import yaml
import numpy as np
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer

import utils

In [ ]:
CORPUS_FILE = 'corpus_2026-10-09.jsonl'
OUT_DIR = 'embeddings'
BATCH_SIZE = 16
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
_cfg = yaml.safe_load(Path("config.yml").read_text())
ST_CONFIGS = _cfg["ST_CONFIGS"]
MEDCPT = _cfg["MEDCPT"]

# Inject configurations and device into the imported utils module
utils.ST_CONFIGS = ST_CONFIGS
utils.MEDCPT = MEDCPT
utils.DEVICE = DEVICE

In [ ]:
def load_corpus(path):
    with open(path) as f:
        records = [json.loads(line) for line in f]
        print(len(records[0]))
    records = [r for r in records[0] if r.get('abstract')]  # drop empty abstracts
    pmids = [r['pubmed_id'] for r in records]
    titles = [r.get('title') or '' for r in records]
    abstracts = [r['abstract'] for r in records]
    docs = [f'{t} {a}'.strip() for t, a in zip(titles, abstracts)]
    return pmids, titles, abstracts, docs


pmids, titles, abstracts, docs = load_corpus(CORPUS_FILE)
print(f'Loaded {len(pmids)} documents | device: {DEVICE}')



100
Loaded 100 documents | device: cuda


In [ ]:
# If a pmids.json already exists (e.g. saved with the BM25 index), make sure
# the order matches, otherwise BM25 and dense results won't line up.
pmids_path = os.path.join(OUT_DIR, 'pmids.json')
if os.path.exists(pmids_path):
    with open(pmids_path) as f:
        saved = json.load(f)
    assert saved == pmids, 'PMID order differs from the saved pmids.json'
    print('PMID order matches existing pmids.json')
else:
    with open(pmids_path, 'w') as f:
        json.dump(pmids, f)


PMID order matches existing pmids.json


In [ ]:
# 2. Truncation check (MedCPT and BGE stop at 512 tokens)

_tok = AutoTokenizer.from_pretrained('BAAI/bge-base-en-v1.5')
lengths = [len(_tok(d)['input_ids']) for d in docs]
n_long = sum(l > 512 for l in lengths)
print(f'Texts over 512 tokens: {n_long} of {len(docs)} '
      f'(max {max(lengths)}, median {int(np.median(lengths))})')
del _tok


def save(key, emb, meta):
    emb = emb.astype('float32')
    np.save(os.path.join(OUT_DIR, f'emb_{key}.npy'), emb)
    meta = {**meta, 'n_docs': len(pmids), 'dim': int(emb.shape[1]),
            'corpus': CORPUS_FILE, 'text': 'title + abstract',
            'normalized': True, 'texts_over_512_tokens': n_long}
    with open(os.path.join(OUT_DIR, f'emb_{key}_meta.json'), 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'Saved emb_{key}.npy  shape={emb.shape}')


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (530 > 512). Running this sequence through the model will result in indexing errors


Texts over 512 tokens: 32 of 100 (max 919, median 462)


In [ ]:
# 3. Sentence-transformers models (BGE, Qwen3)
for key, cfg in ST_CONFIGS.items():
    print(f'\nEmbedding with {cfg["name"]} ...')
    model = SentenceTransformer(cfg['name'], device=DEVICE)
    model.max_seq_length = cfg['max_length']
    emb = model.encode(docs, batch_size=BATCH_SIZE, normalize_embeddings=True,
                       show_progress_bar=True, convert_to_numpy=True)
    save(key, emb, cfg)
    del model
    free_memory()




Embedding with BAAI/bge-base-en-v1.5 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Saved emb_qwen3.npy  shape=(100, 1024)


In [ ]:
#4. MedCPT article encoder#
print(f'\nEmbedding with {MEDCPT["article_encoder"]} ...')
tok = AutoTokenizer.from_pretrained(MEDCPT['article_encoder'])
model = AutoModel.from_pretrained(MEDCPT['article_encoder']).to(DEVICE).eval()

pairs = [[t, a] for t, a in zip(titles, abstracts)]
chunks = []
with torch.no_grad():
    for i in range(0, len(pairs), BATCH_SIZE):
        enc = tok(pairs[i:i + BATCH_SIZE], truncation=True, padding=True,
                  max_length=MEDCPT['article_max_length'],
                  return_tensors='pt').to(DEVICE)
        cls = model(**enc).last_hidden_state[:, 0, :]
        chunks.append(torch.nn.functional.normalize(cls, dim=-1).cpu())
emb = torch.cat(chunks).numpy()
save('medcpt', emb, MEDCPT)
del model, tok
free_memory()



Embedding with ncbi/MedCPT-Article-Encoder ...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved emb_medcpt.npy  shape=(100, 768)


In [ ]:
# 5. Sanity check: top results per model and how much the models agree

TEST_QUERIES = [
    'hallucination in radiology report generation by vision-language models',
    'GPT-4o accuracy on medical visual question answering benchmarks',
    'comparison of multimodal models with pathologists on histopathology images',
]

top10 = {}
for key in ['medcpt', 'bge', 'qwen3']:
    doc_emb = np.load(os.path.join(OUT_DIR, f'emb_{key}.npy'))
    qenc = utils.QueryEncoder(key)
    q_emb = qenc.encode(TEST_QUERIES)
    scores = q_emb @ doc_emb.T  # cosine similarity (all vectors normalized)
    top10[key] = []
    print(f'\n=== {key} ===')
    for qi, q in enumerate(TEST_QUERIES):
        order = np.argsort(-scores[qi])[:10]
        top10[key].append({pmids[j] for j in order})
        print(f'\nQ: {q}')
        for rank, j in enumerate(order[:5], 1):
            print(f'  {rank}. [{pmids[j]}] {scores[qi, j]:.3f}  {titles[j][:90]}')
    del qenc
    free_memory()

print('\nTop-10 overlap between models (averaged over test queries):')
keys = list(top10)
for a in range(len(keys)):
    for b in range(a + 1, len(keys)):
        ka, kb = keys[a], keys[b]
        overlap = np.mean([len(x & y) / 10 for x, y in zip(top10[ka], top10[kb])])
        print(f'  {ka} vs {kb}: {overlap:.0%}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== medcpt ===

Q: hallucination in radiology report generation by vision-language models
  1. [42649333] 0.679  Do Accuracy Gains Reflect Genuine Visual Understanding? A Multi-Model Evaluation of Vision
  2. [42767241] 0.655  Artificial Intelligence in Radiology: Methodological Foundations, Clinical Applications, a
  3. [42582763] 0.654  Swin-Qwen3: a three-stage vision-language framework for automated radiology report generat
  4. [42585107] 0.650  Leveraging large language models to enhance cytopathology: Opportunities, challenges, and 
  5. [42771632] 0.639  Foundation Models in Ophthalmic Artificial Intelligence: Current Status and Future Directi

Q: GPT-4o accuracy on medical visual question answering benchmarks
  1. [42649336] 0.687  Generating Image-Based Multiple-Choice Questions with Multimodal Large Language Models: Ex
  2. [42649333] 0.655  Do Accuracy Gains Reflect Genuine Visual Understanding? A Multi-Model Evaluation of Vision
  3. [42718906] 0.654  Large Language Model

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== bge ===

Q: hallucination in radiology report generation by vision-language models
  1. [42582763] 0.791  Swin-Qwen3: a three-stage vision-language framework for automated radiology report generat
  2. [42723869] 0.767  Bridging semantics and clinical fidelity: a section-based assessment of a vision-language 
  3. [42645937] 0.759  An Explainable Multimodal Framework for Breast Ultrasound Report Generation Using Vision-L
  4. [42818963] 0.754  An Automated, Contamination-Controlled VQA Benchmark for Evaluating Vision-Language Models
  5. [42630590] 0.753  Cross-modal bias in medical vision-language models: a pipeline-aware framework for mechani

Q: GPT-4o accuracy on medical visual question answering benchmarks
  1. [42694118] 0.798  Comparative performance of contemporary multimodal large language models in retinal imagin
  2. [42818963] 0.782  An Automated, Contamination-Controlled VQA Benchmark for Evaluating Vision-Language Models
  3. [42649336] 0.766  Generating Image-Based 

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


=== qwen3 ===

Q: hallucination in radiology report generation by vision-language models
  1. [42649333] 0.678  Do Accuracy Gains Reflect Genuine Visual Understanding? A Multi-Model Evaluation of Vision
  2. [42573684] 0.663  CONRep: Uncertainty-Aware Vision-Language Report Drafting Using Conformal Prediction.
  3. [42582763] 0.660  Swin-Qwen3: a three-stage vision-language framework for automated radiology report generat
  4. [42716901] 0.658  AI in medical imaging: Practical vision-language model prompt engineering for radiologists
  5. [42723869] 0.642  Bridging semantics and clinical fidelity: a section-based assessment of a vision-language 

Q: GPT-4o accuracy on medical visual question answering benchmarks
  1. [42649333] 0.602  Do Accuracy Gains Reflect Genuine Visual Understanding? A Multi-Model Evaluation of Vision
  2. [42649336] 0.597  Generating Image-Based Multiple-Choice Questions with Multimodal Large Language Models: Ex
  3. [42694118] 0.594  Comparative performance of